# DETR Physical Patch Attack & Defense — Proof of Concept

**What this notebook demonstrates:**
1. Load a pretrained DETR object detector and run normal detection.
2. Train a simple adversarial patch (with EOT-lite: random rotation, scale, position, brightness) that suppresses DETR's detections.
3. Build a simple **explainable defense**: a gradient-based saliency check that flags *where* and *why* an input looks suspicious.
4. Measure the defense's extra inference cost, since real-time use is part of the goal.

**Important — read before you present this anywhere:** this is a *learning / proof-of-concept* notebook, not a research result. Section 8 at the bottom spells out exactly why, and what would need to change before this counts as a real contribution. Please read that section as carefully as the rest.

**Before running:** this needs internet access to download DETR's pretrained weights from Hugging Face, so run this in Google Colab (free GPU, File > Save a copy to Drive) or a machine with internet + a GPU. It will run on CPU too, just slower.

## Step 0: Setup

In [ ]:
!pip install -q transformers torch torchvision pillow matplotlib requests

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import DetrImageProcessor, DetrForObjectDetection

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Step 1: Load pretrained DETR

`facebook/detr-resnet-50` is the original DETR model, pretrained on COCO. This is also the exact model most attack papers (e.g. Nazeri et al.) evaluate against, so results here are comparable to what you'll read in the literature.

In [ ]:
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
model.to(device)
model.eval()

MEAN = torch.tensor(processor.image_mean, device=device).view(1, 3, 1, 1)
STD = torch.tensor(processor.image_std, device=device).view(1, 3, 1, 1)
IMG_SIZE = (600, 800)  # (H, W) -- fixed size for this demo, simplified vs. DETR's default resizing

def normalize(img01):
    return (img01 - MEAN) / STD

## Step 2: Get a sample image

The line below uses a demo COCO image just to confirm the pipeline works end to end.

**Swap in your own driving image (KITTI/BDD100K frame) for this to actually mean something for your research** — uncomment the second option below and set the path after uploading a file to Colab (folder icon on the left sidebar).

In [ ]:
def load_image_as_tensor(source, is_url=True):
    if is_url:
        resp = requests.get(source, timeout=10)
        img = Image.open(BytesIO(resp.content)).convert("RGB")
    else:
        img = Image.open(source).convert("RGB")
    img = img.resize((IMG_SIZE[1], IMG_SIZE[0]))
    arr = np.array(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)  # 1,3,H,W
    return tensor, img

DEMO_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image_tensor, pil_image = load_image_as_tensor(DEMO_URL, is_url=True)

# To use your own AV image instead (recommended):
# image_tensor, pil_image = load_image_as_tensor("/content/your_driving_image.jpg", is_url=False)

## Step 3: Baseline detection

Run DETR on the clean image and visualize its detections, before any attack.

In [ ]:
def detect(image01, threshold=0.7):
    with torch.no_grad():
        inputs = normalize(image01)
        outputs = model(pixel_values=inputs)
    target_sizes = torch.tensor([IMG_SIZE], device=device)
    results = processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=threshold
    )[0]
    return results

def show_detections(image01, results, title):
    img_np = image01.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(np.clip(img_np, 0, 1))
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        x0, y0, x1, y1 = box.tolist()
        rect = mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2,
                                   edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        label_name = model.config.id2label[label.item()]
        ax.text(x0, y0 - 5, f"{label_name}: {score:.2f}", color="lime", fontsize=9,
                 bbox=dict(facecolor="black", alpha=0.5, pad=1))
    ax.set_title(title)
    ax.axis("off")
    plt.show()

baseline_results = detect(image_tensor)
show_detections(image_tensor, baseline_results, "Baseline detections (no attack)")
print(f"Baseline: {len(baseline_results['scores'])} objects detected")

## Step 4: Train a simple adversarial patch (attack)

This builds a small learnable patch, then repeatedly:
- pastes it onto the image at a **random rotation, scale, position, and brightness** (this randomization across transforms is the core idea of **EOT — Expectation over Transformation** — it's what makes a patch survive being seen from different real-world angles/distances instead of only working in one exact digital position),
- runs DETR on the result,
- pushes the patch's pixels (via gradient descent) to reduce DETR's confidence that *anything* is an object.

This is a simplified, untargeted "hide the objects" attack — weaker and less refined than what's in the published literature (Nazeri et al., Wei et al. use far more steps and more careful loss functions). It's here to make the concept concrete, not to match state-of-the-art attack strength.

In [ ]:
PATCH_SIZE = 100
patch = torch.rand(3, PATCH_SIZE, PATCH_SIZE, device=device, requires_grad=True)

def place_patch(canvas_size, patch, angle_rad, scale, tx, ty):
    C, ph, pw = patch.shape
    H, W = canvas_size
    canvas = torch.zeros(1, C, H, W, device=patch.device)
    mask = torch.zeros(1, 1, H, W, device=patch.device)
    top, left = (H - ph) // 2, (W - pw) // 2
    canvas[0, :, top:top + ph, left:left + pw] = patch
    mask[0, :, top:top + ph, left:left + pw] = 1.0

    cos_a, sin_a = torch.cos(angle_rad), torch.sin(angle_rad)
    theta = torch.stack([
        torch.stack([scale * cos_a, -scale * sin_a, tx]),
        torch.stack([scale * sin_a,  scale * cos_a, ty]),
    ]).unsqueeze(0).to(patch.device)

    grid = F.affine_grid(theta, canvas.size(), align_corners=False)
    canvas_t = F.grid_sample(canvas, grid, align_corners=False)
    mask_t = F.grid_sample(mask, grid, align_corners=False)
    return canvas_t, mask_t

def apply_eot_patch(image01, patch):
    angle = torch.empty(1, device=patch.device).uniform_(-20, 20) * (np.pi / 180)
    scale = torch.empty(1, device=patch.device).uniform_(0.8, 1.2)
    tx = torch.empty(1, device=patch.device).uniform_(-0.6, 0.6)
    ty = torch.empty(1, device=patch.device).uniform_(-0.6, 0.6)
    brightness = torch.empty(1, device=patch.device).uniform_(0.8, 1.2)

    canvas, mask = place_patch(IMG_SIZE, patch, angle[0], scale[0], tx[0], ty[0])
    patched = image01 * (1 - mask) + torch.clamp(canvas * brightness, 0, 1) * mask
    return patched

def attack_loss(image01):
    """Untargeted 'hide the objects' loss: push detections toward the no-object class."""
    outputs = model(pixel_values=normalize(image01))
    probs = outputs.logits.softmax(-1)
    no_obj_idx = outputs.logits.shape[-1] - 1
    foreground_conf = 1 - probs[..., no_obj_idx]
    return foreground_conf.mean()

optimizer = torch.optim.Adam([patch], lr=0.05)
N_STEPS = 60  # kept small for a quick demo; real attacks typically use hundreds-thousands of steps

print("Training adversarial patch (EOT-lite)...")
for step in range(N_STEPS):
    optimizer.zero_grad()
    patched_image = apply_eot_patch(image_tensor, patch)
    loss = attack_loss(patched_image)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        patch.clamp_(0, 1)
    if step % 10 == 0 or step == N_STEPS - 1:
        print(f"  step {step:3d} | mean foreground confidence = {loss.item():.4f}")

## Step 5: See the attack's effect

Apply the trained patch and compare detections to the baseline.

In [ ]:
with torch.no_grad():
    final_patched = apply_eot_patch(image_tensor, patch)
attacked_results = detect(final_patched)
show_detections(final_patched, attacked_results, "Detections WITH adversarial patch")
print(f"After patch: {len(attacked_results['scores'])} objects detected "
      f"(baseline had {len(baseline_results['scores'])})")

## Step 6: Defense — explainable, saliency-based patch flagging

The idea: an adversarial patch has to create a large, concentrated effect on the model's output using a small region of pixels. That leaves a signature — the model's output becomes **unusually sensitive** to that specific region. We can measure "sensitivity" as the gradient of the output with respect to each pixel (a **saliency map**), then look for a small region whose gradient magnitude is way above the image's average.

This doubles as the **explainability** piece: instead of just saying "adversarial input detected," we show *which region* triggered the flag, and *why* (unusually high local gradient concentration) — this is the same category of idea as the ICEv2 patch-attribution approach you've already studied, just implemented at the simplest possible level here.

Note this is a simple heuristic, not a rigorous method — see the limitations at the end.

In [ ]:
def compute_saliency(image01):
    img = image01.clone().detach().requires_grad_(True)
    outputs = model(pixel_values=normalize(img))
    probs = outputs.logits.softmax(-1)
    no_obj_idx = outputs.logits.shape[-1] - 1
    foreground_conf = 1 - probs[..., no_obj_idx]
    score = foreground_conf.mean()
    model.zero_grad()
    score.backward()
    saliency = img.grad.abs().sum(dim=1, keepdim=True)
    return saliency.detach()

def flag_suspect_region(saliency, window=64, stride=32):
    pooled = F.avg_pool2d(saliency, kernel_size=window, stride=stride)
    flat_idx = torch.argmax(pooled)
    w_cells = pooled.shape[-1]
    row, col = divmod(flat_idx.item(), w_cells)
    y, x = row * stride, col * stride
    peak = pooled.max().item()
    mean_val = saliency.mean().item()
    anomaly_ratio = peak / (mean_val + 1e-8)
    return x, y, window, anomaly_ratio

def show_defense_result(image01, x, y, window, anomaly_ratio, threshold=6.0):
    img_np = image01.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(np.clip(img_np, 0, 1))
    rect = mpatches.Rectangle((x, y), window, window, linewidth=2.5,
                               edgecolor="red", facecolor="none", linestyle="--")
    ax.add_patch(rect)
    verdict = "SUSPICIOUS -- likely adversarial patch" if anomaly_ratio > threshold else "Looks clean"
    ax.set_title(f"Defense flag: {verdict}  (anomaly ratio={anomaly_ratio:.1f}, threshold={threshold})")
    ax.axis("off")
    plt.show()
    print(f"Explanation: this region had {anomaly_ratio:.1f}x the average gradient "
          f"magnitude in the image, i.e. the model's output was unusually sensitive "
          f"to these pixels -- the signature this demo uses for 'why flagged'.")

saliency_map = compute_saliency(final_patched)
x, y, window, anomaly_ratio = flag_suspect_region(saliency_map)
show_defense_result(final_patched, x, y, window, anomaly_ratio)

# Sanity check: run the same defense on the clean baseline image (should NOT flag it)
clean_saliency = compute_saliency(image_tensor)
cx, cy, cwindow, clean_ratio = flag_suspect_region(clean_saliency)
print(f"\nSanity check on clean image -- anomaly ratio: {clean_ratio:.1f} "
      f"(should be much lower than the attacked image's {anomaly_ratio:.1f})")

## Step 7: Efficiency check (real-time constraint)

Your guide's "efficient" requirement means the defense can't be expensive to run. This measures the extra time the saliency check adds on top of normal detection.

In [ ]:
def time_it(fn, n=20):
    for _ in range(3):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(n):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - start) / n

baseline_time = time_it(lambda: detect(image_tensor))
defended_time = time_it(lambda: flag_suspect_region(compute_saliency(image_tensor)))

print(f"Baseline inference:            {baseline_time*1000:.1f} ms/image ({1/baseline_time:.1f} FPS)")
print(f"Detection + defense overhead:  {defended_time*1000:.1f} ms/image ({1/defended_time:.1f} FPS)")
print(f"Extra cost of defense:         {(defended_time - baseline_time)*1000:.1f} ms/image")

## Step 8: Limitations — what this notebook is *not*, and what real research would need

Read this before showing the notebook to your guide, so you can frame it correctly.

1. **The attack here is intentionally weak and simplified.** It's an untargeted "suppress everything" attack over 60 optimization steps. Published attacks (Nazeri et al., Wei et al.) use far more steps, more carefully designed loss functions, and often target specific objects/classes rather than everything at once.
2. **The EOT here is "EOT-lite."** Real EOT implementations sample from much larger, more realistic transform distributions (varying distance, full 3D viewpoint, real lighting/weather conditions, sometimes real camera capture-and-recapture), not just 2D rotation/scale/position/brightness.
3. **The defense has NOT been adaptively evaluated.** This is the most important limitation. An attacker who *knows* this saliency-based defense exists could optimize the patch to simultaneously (a) fool the detector and (b) keep its saliency signature low and spread out, defeating this exact check. Until that adaptive test is run, this defense's apparent success is not trustworthy — this is precisely the Athalye et al. standard you've applied when critiquing other papers.
4. **No quantitative, dataset-scale evaluation.** This is a single-image qualitative demo. Real evaluation needs attack success rate / defense success rate across many images from KITTI or BDD100K, with proper metrics, not just "it worked once."
5. **The saliency approach itself is a simple heuristic**, not a rigorous explainability method — it's a starting point to build from, potentially connecting to the more careful patch-attribution approach in ICEv2, not an endpoint.

**What to say to your guide about this notebook:** "This demonstrates the mechanism I want to research works end-to-end, but the attack, the defense, and the evaluation all need to be made rigorous — adaptive attack testing, dataset-scale evaluation, and a stronger explainability method are the next steps."